<a href="https://colab.research.google.com/github/LiubovRev/Applied-ML-Case-Studies/blob/main/vision_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


class PatchEmbed(nn.Module):
    def __init__(self, patch_size=16, embed_dim=768):
        super().__init__()
        self.proj = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)       # (B, 768, 14, 14)
        x = x.flatten(2)       # (B, 768, 196)
        x = x.transpose(1, 2)  # (B, 196, 768)
        return x


class ViT(nn.Module):
    def __init__(self, n_classes=10, embed_dim=768, n_heads=12, depth=12):
        super().__init__()
        n_patches = 196

        self.patch_embed = PatchEmbed(patch_size=16, embed_dim=embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed  = nn.Parameter(torch.zeros(1, 1 + n_patches, embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=embed_dim * 4,
            activation='gelu',
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, n_classes)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embed
        x = self.transformer(x)
        x = self.norm(x[:, 0])   # only CLS token
        return self.head(x)


# --- Quick Check ---
if __name__ == "__main__":
    model = ViT(n_classes=10)
    img = torch.randn(2, 3, 224, 224)
    out = model(img)
    print("Output shape:", out.shape)  # torch.Size([2, 10])

    # --- Training on CIFAR-10 ---
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

    train_data = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    loader = DataLoader(train_data, batch_size=32, shuffle=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(5):
        for imgs, labels in loader:
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch + 1} done")